**Sentiment Analysis using NLP Pipeline & ML Models**



**TASK 1: Data Understanding**

In [13]:
import pandas as pd

df = pd.read_csv("twitter.csv", encoding='latin-1')

print("First 5 rows:\n", df.head())

# Convert labels
df['sentiment'] = df['label'].replace({
    0: 'negative',
    1: 'positive'
})

# Balance dataset
df_neg = df[df['sentiment'] == 'negative'].sample(2000, random_state=42)
df_pos = df[df['sentiment'] == 'positive'].sample(2000, random_state=42)

df = pd.concat([df_neg, df_pos])

# Keep needed columns
df = df[['tweet', 'sentiment']]
df.rename(columns={'tweet': 'text'}, inplace=True)

print("\nTotal Samples:", df.shape[0])
print("\nClass Distribution:\n", df['sentiment'].value_counts())

print("\nSample Text:\n", df['text'].head())

First 5 rows:
    id  label                                              tweet
0   1      0   @user when a father is dysfunctional and is s...
1   2      0  @user @user thanks for #lyft credit i can't us...
2   3      0                                bihday your majesty
3   4      0  #model   i love u take with u all the time in ...
4   5      0             factsguide: society now    #motivation

Total Samples: 4000

Class Distribution:
 sentiment
negative    2000
positive    2000
Name: count, dtype: int64

Sample Text:
 8824     #body to body massage with a   ending oil #mas...
31854     @user @ my call back!  #casting #castingcall ...
28079    help creates the #environment of #togetherness...
29214    summer with friendÃ¢ÂÂ¨Ã°ÂÂÂ¥ #summer  #fri...
20025    follow me on snapchat at awesomecutenes7 #snap...
Name: text, dtype: object




**TASK 2: NLP Preprocessing**

In [14]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove special characters & punctuation
    text = re.sub(r"[^a-zA-Z]", " ", text)

    # Tokenization
    words = text.split()

    # Remove stopwords
    words = [w for w in words if w not in stop_words]

    # Stemming
    words = [stemmer.stem(w) for w in words]

    return " ".join(words)

df['clean_text'] = df['text'].apply(preprocess)

df.head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,text,sentiment,clean_text
8824,#body to body massage with a ending oil #mas...,negative,bodi bodi massag end oil massag bodi happyend ...
31854,@user @ my call back! #casting #castingcall ...,negative,user call back cast castingcal model cute todd...
28079,help creates the #environment of #togetherness...,negative,help creat environ togeth amp mutualrespect pr...
29214,summer with friendÃ¢ÂÂ¨Ã°ÂÂÂ¥ #summer #fri...,negative,summer friend summer friend life vlog weeknd c...
20025,follow me on snapchat at awesomecutenes7 #snap...,negative,follow snapchat awesomecuten snapchat selfi sa...


**TASK 3: Feature Engineering**

*   Bag of Words (BoW)
*   TF-IDF



In [16]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(max_features=5000)

X_bow = bow.fit_transform(df['clean_text'])

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X_tfidf = tfidf.fit_transform(df['clean_text'])

**TASK 4: Model Building**

*   Train-Test Split
*   Logistic Regression


*   Naive Bayes

*   Decision Tree






In [18]:
from sklearn.model_selection import train_test_split

y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

In [19]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(class_weight='balanced', max_iter=200)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

In [20]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

In [21]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

**TASK 5: Model Evaluation**

In [22]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(y_test, y_pred, name):
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, pos_label='positive'))
    print("Recall:", recall_score(y_test, y_pred, pos_label='positive'))
    print("F1 Score:", f1_score(y_test, y_pred, pos_label='positive'))

evaluate(y_test, y_pred_lr, "Logistic Regression")
evaluate(y_test, y_pred_nb, "Naive Bayes")
evaluate(y_test, y_pred_dt, "Decision Tree")


Logistic Regression
Accuracy: 0.87375
Precision: 0.8568019093078759
Recall: 0.8975
F1 Score: 0.8766788766788767

Naive Bayes
Accuracy: 0.88125
Precision: 0.8692493946731235
Recall: 0.8975
F1 Score: 0.8831488314883149

Decision Tree
Accuracy: 0.785
Precision: 0.8149171270718232
Recall: 0.7375
F1 Score: 0.7742782152230971


**TASK 6: Comparison & Insights**

                                                             
                                                         
                                                                                        
                

*   TF-IDF performed better than Bag of Words because it captures
important words more effectively.
*   Logistic Regression gave the highest accuracy and balanced performance.

*   Naive Bayes worked well for text classification but slightly lower than Logistic Regression.
*   Decision Tree showed lower performance due to overfitting.   


* Preprocessing steps like stopword removal and stemming improved the results.

  

*  Balancing the dataset helped avoid bias toward negative class.
*  There is a trade-off between model complexity and performance, where simpler models like Logistic Regression performed better.


